Loading into spark dataframe

In [0]:
%python

cust_info = spark.read.table('workspace.bronze.crm_cust_info')

Drop Null customer ids and remove duplicates by choosing the latest entry

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

#drop null customer ids
cust_info = cust_info.dropna(subset='cst_id')

#drop duplicate customer id rows; most recent entry is kept
window_spec = Window.partitionBy('cst_id').orderBy(col('cst_create_date').desc())

cust_info_deduped = (
    cust_info.withColumn('rn', row_number().over(window_spec))
    .filter(col('rn') == 1)
    .drop('rn')
)

Gender & Marital Status Column Transformations

In [0]:
from pyspark.sql.functions import when, col
#
df = (
    cust_info_deduped.withColumn('cst_marital_status',    
        when((col('cst_marital_status')) == 'S', "Single")
        .when((col('cst_marital_status')) == 'M', "Married")
        .otherwise("Unknown")
    )
    .withColumn('cst_gndr',    
        when((col('cst_gndr')) == 'M', "Male")
        .when((col('cst_gndr')) == 'F', "Female")
        .otherwise("Unknown")
))
df.display()

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col


for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()